# Faza 4 - dokonczenie wariantu E (LoRA), tylko trening

Kolejnosc:
1. Setup: clone repo, lekkie deps, Pets
2. HF auth + pull `variant_E.zip` + pull manifestu osobno (ten w zipie byl wadliwy)
3. wgrywanie zipow z poprzednich, niedokonczonych seedow
4. Petla treningu E configow

## 1. Repo + lekkie deps + Pets

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q timm==1.0.11 transformers==4.46.3 huggingface_hub==0.26.2 PyYAML==6.0.2

In [ ]:
from pathlib import Path
PETS = Path('data/raw/oxford-iiit-pet/images')
if not (PETS.exists() and any(PETS.iterdir())):
    !python scripts/download_pets.py

## 2. HF auth + pull variant_E + manifest

In [ ]:
from huggingface_hub import login, hf_hub_download, whoami
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('brak HF_TOKEN - dodaj w Colab Secrets')
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in as:', whoami()['name'])

HF_REPO_ID = 'micwuj/dlicv-synth'

In [ ]:
import zipfile, shutil
VARIANT_E = Path('data/synthetic/variant_E')
MANIFEST = VARIANT_E / 'manifest_kept.csv'
sample_png = VARIANT_E / 'Ragdoll' / 'Ragdoll_0000.png'

if not sample_png.exists():
    zip_path = hf_hub_download(
        repo_id=HF_REPO_ID, repo_type='dataset',
        filename='variant_E.zip', local_dir='data/synthetic',
    )
    VARIANT_E.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(VARIANT_E)
    # auto-flatten - na wypadek gdyby zip mial prefiks variant_E/
    nested = VARIANT_E / 'variant_E'
    if nested.exists() and nested.is_dir():
        for d in nested.iterdir():
            tgt = VARIANT_E / d.name
            if tgt.exists():
                shutil.rmtree(tgt)
            shutil.move(str(d), str(tgt))
        nested.rmdir()
    print(f'rozpakowane pngs: {sum(1 for _ in VARIANT_E.rglob("*.png"))}')
else:
    print(f'variant_E juz rozpakowane ({sum(1 for _ in VARIANT_E.rglob("*.png"))} pngs).')

# Manifest w samym zipie jest 0 bajtow (bug starej sesji) - dociagamy osobno
hf_hub_download(
    repo_id=HF_REPO_ID, repo_type='dataset',
    filename='variant_E/manifest_kept.csv', local_dir='data/synthetic',
)
with open(MANIFEST) as f:
    n_rows = sum(1 for _ in f) - 1
print(f'manifest: {n_rows} rows')
assert n_rows > 2000, f'manifest looks broken: {n_rows} rows'
assert sample_png.exists(), f'missing sample png: {sample_png}'

## 3. Wgranie outputs z poprzednich, niedokonczonych seedow

In [ ]:
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    if not fname.endswith('.zip'):
        continue
    with zipfile.ZipFile(fname) as zf:
        names = zf.namelist()
        zf.extractall('.' if any(n.startswith('outputs/') for n in names) else 'outputs')
    os.remove(fname)
    print(f'unpacked {fname}')

existing = sorted(Path('outputs').glob('E_*/final_results.json'))
print(f'\nGotowe runy w outputs/: {len(existing)}')
for p in existing:
    print(f'  {p.parent.name}')

## 4. Petla treningu z zip+download po kazdym seedzie

In [ ]:
!python scripts/generate_configs.py

In [ ]:
import subprocess, time, sys
from google.colab import files

EXP_DIR = Path('configs/exp')
BASE = 'configs/base.yaml'
OVERRIDES = ['configs/colab.yaml']
OUTPUTS = Path('outputs')

configs = sorted(EXP_DIR.glob('E_*.yaml'))
runnable = [c for c in configs if not (OUTPUTS / c.stem / 'final_results.json').exists()]
skipped = [c for c in configs if c not in runnable]

print(f'E configow lacznie: {len(configs)}')
print(f'Pomijam (juz wytrenowane): {len(skipped)}')
for c in skipped:
    print(f'  {c.stem}')
print(f'Do uruchomienia: {len(runnable)}')
for c in runnable:
    print(f'  {c.stem}')

total_start = time.time()
for i, cfg in enumerate(runnable, 1):
    run_name = cfg.stem
    elapsed_total = (time.time() - total_start) / 60
    print(f'\n{"=" * 72}')
    print(f'[{i}/{len(runnable)}] {run_name}  (total elapsed: {elapsed_total:.1f} min)')
    print(f'{"=" * 72}')
    cmd = [sys.executable, '-m', 'src.train', '--config', BASE, *OVERRIDES, str(cfg)]
    start = time.time()
    rc = subprocess.run(cmd).returncode
    print(f'\n[{run_name}] rc={rc} elapsed={time.time() - start:.0f}s')
    if rc != 0:
        print('przerwane - sprawdz traceback wyzej w outpucie komorki')
        break
    zip_base = f'/content/{run_name}'
    shutil.make_archive(zip_base, 'zip', OUTPUTS, run_name)
    files.download(f'{zip_base}.zip')
    print(f'zip+download: {zip_base}.zip')

print(f'\nDONE total: {(time.time() - total_start) / 60:.1f} min')

## 5. Final dump + aggregate

In [ ]:
shutil.make_archive('/content/outputs_E_full', 'zip', '.', 'outputs')
files.download('/content/outputs_E_full.zip')

In [ ]:
!python scripts/aggregate_results.py
import pandas as pd
pd.read_csv('outputs/_aggregate/summary_by_model_variant.csv')